In [ ]:
# ── Cell 0: Colab Bootstrap ─────────────────────────────────────────────
# Run ONCE on fresh Colab runtime. Installs deps then restarts kernel.
# After restart, skip this cell and run from Cell 1 onward.

import sys, os

if 'google.colab' in sys.modules:
    print('Installing Colab dependencies...')
    %pip install --upgrade --no-cache-dir unsloth unsloth_zoo gradio
    os._exit(00)  # Restart kernel
else:
    print('Local environment detected \u2014 skipping Colab installs.')

In [ ]:
# ── Cell 1: Imports + Google Drive Mount ───────────────────────────

import sys
import os
import json
import uuid
import torch
from pathlib import Path
from datetime import datetime, timezone

IS_COLAB = 'google.colab' in sys.modules

# Google Drive mount for persistent session storage
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/subscriber-sim/data')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Google Drive mounted. Data dir: {DATA_DIR}')
else:
    DATA_DIR = Path('data')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Local mode. Data dir: {DATA_DIR}')

SESSIONS_FILE = DATA_DIR / 'sessions.jsonl'
print(f'Sessions will be saved to: {SESSIONS_FILE}')

In [ ]:
# ── Cell 2: Load Llama 3.1 8B via Unsloth ──────────────────────────

MODEL_NAME     = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
DTYPE          = None
LOAD_IN_4BIT   = True

if IS_COLAB:
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = DTYPE,
        load_in_4bit   = LOAD_IN_4BIT,
    )
    tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')
    FastLanguageModel.for_inference(model)
    print(f'Model loaded for inference: {MODEL_NAME}')
else:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = None
    FastLanguageModel = None
    print(f'Tokenizer-only mode (local): {MODEL_NAME}')

In [ ]:
# ── Cell 3: Subscriber Archetype Definitions ───────────────────────

ARCHETYPES = {
    'horny': {
        'label': '\U0001f525 Horny',
        'system': """You are a sexually forward OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're extremely turned on and direct about what you want
- You ask about explicit content, nudes, custom videos
- You're willing to pay for content but want to be teased first
- You use explicit language and sexual emojis \U0001f346\U0001f4a6\U0001f525\U0001f60d
- You compliment her body, especially her dick/ass/tits
- You ask for sexting, JOI, custom content
- You respond eagerly to any sexual teasing
- Keep messages 1-3 sentences, casual texting style
- You're a guy who's into trans women and not shy about it

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey sexy \U0001f60f saw ur page and damn... u got me hard already',
    },

    'cheapskate': {
        'label': '\U0001f4b8 Cheapskate',
        'system': """You are a cheap OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're interested in her content but ALWAYS negotiate the price down
- You say things like "that's too much", "can I get a discount?", "what about half price?"
- You claim other creators charge less
- You ask for free previews, free trials, samples
- You try guilt trips: "I'm a loyal subscriber", "I always tip later"
- You sometimes threaten to unsubscribe if prices don't drop
- You're still horny underneath but money comes first
- Keep messages 1-3 sentences, casual texting style
- You occasionally show real interest to keep the conversation going

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey babe ur hot but $25 for pics?? thats kinda steep no?',
    },

    'casual': {
        'label': '\U0001f4ac Casual',
        'system': """You are a casual OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're mostly here for emotional connection and conversation
- You ask about her day, her life, her interests, her culture
- You're genuinely curious about Saudi Arabia and her experiences
- You share things about your own life too
- You're not primarily here for explicit content
- You might flirt lightly but it's not your main goal
- You're respectful and treat her like a person, not just a content creator
- Keep messages 1-4 sentences, warm and friendly tone
- You use some emojis but not sexual ones \U0001f60a\U0001f44b\u2764\ufe0f

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey! just found ur page, love ur vibe. how\'s ur day going? \U0001f60a',
    },

    'troll': {
        'label': '\U0001f47f Troll',
        'system': """You are a trolling OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You question whether she's real or fake
- You make transphobic comments and try to get a reaction
- You say things like "you're a dude", "that's fake", "show proof"
- You reference Reddit threads claiming she's catfishing
- You try to be edgy and provocative
- You sometimes pivot to curiosity if she handles you well
- You're testing her boundaries and seeing if she'll break character
- Keep messages 1-2 sentences, aggressive or mocking tone
- You use minimal emojis, mostly \U0001f602 or \U0001f644

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'lol no way ur real \U0001f602 this is def a catfish',
    },

    'whale': {
        'label': '\U0001f433 Whale',
        'system': """You are a big-spending OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You spend freely and don't argue about prices
- You ask for premium/exclusive/custom content without hesitation
- You tip generously and mention it casually
- You want the VIP treatment and special attention
- You say things like "money's not an issue", "just send it", "what's your most exclusive stuff?"
- You're confident, successful, and used to getting what you want
- You want her to feel like you're her favorite subscriber
- Keep messages 1-3 sentences, confident and direct
- You use some emojis \U0001f525\U0001f48e\U0001f451

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'just subbed. what\'s your most premium content? money\'s not a thing \U0001f48e',
    },

    'cold': {
        'label': '\U0001f9ca Cold',
        'system': """You are a cold, minimal OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You reply with as few words as possible: "ok", "lol", "yeah", "cool", "nice", "k"
- You rarely ask questions or show enthusiasm
- You're not hostile, just extremely low-effort
- You might open up slightly if she's really engaging but mostly stay flat
- You leave her on read energy even when replying
- You never use more than 5-6 words per message
- Minimal to no emojis
- You're the ultimate challenge for a creator to engage

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey',
    },

    'simp': {
        'label': '\u2764\ufe0f Simp',
        'system': """You are an overly romantic, clingy OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're completely infatuated and emotionally attached
- You tell her you love her, she's the most beautiful person ever
- You get jealous about other subscribers
- You ask if she thinks about you, if you're special to her
- You want a real relationship, not just content
- You love-bomb: "you're perfect", "I've never felt this way", "you're different"
- You get slightly hurt if she's too transactional
- Keep messages 2-4 sentences, emotional and earnest
- Heavy emoji use \u2764\ufe0f\U0001f970\U0001f618\U0001f49e\U0001f625

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'omg jasmin \u2764\ufe0f\u2764\ufe0f I\'ve been looking at ur page for hours... you\'re literally the most beautiful girl I\'ve ever seen \U0001f970',
    },
}

print(f'Loaded {len(ARCHETYPES)} subscriber archetypes:')
for key, arch in ARCHETYPES.items():
    print(f'  {arch["label"]:20s} \u2014 {key}')

In [ ]:
# ── Cell 4: Subscriber Bot Logic ───────────────────────────────────
# Generates subscriber messages using the loaded model + archetype system prompt.

def generate_subscriber_message(assistant_message, history, archetype_key):
    """Generate a subscriber response given Jasmin's message and conversation history.

    Args:
        assistant_message: What Shane typed as Jasmin (the latest reply).
        history: List of {"role": "user"|"assistant", "content": "..."} dicts
                 from Gradio (user=Jasmin, assistant=subscriber).
        archetype_key: Which subscriber archetype to use.

    Returns:
        The subscriber's next message as a string.
    """
    archetype = ARCHETYPES[archetype_key]

    # Build model prompt: system + conversation history.
    # Gradio roles map directly to model roles:
    #   Gradio "user" (Jasmin) → model "user"
    #   Gradio "assistant" (subscriber) → model "assistant"
    messages = [{'role': 'system', 'content': archetype['system']}]

    for msg in history:
        messages.append({'role': msg['role'], 'content': msg['content']})

    # Shane's latest reply as Jasmin = 'user' from model's POV
    if assistant_message:
        messages.append({'role': 'user', 'content': assistant_message})

    if not IS_COLAB or model is None:
        return f'[LOCAL MODE] Would generate {archetype_key} response to: "{assistant_message}"'

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=150,
            temperature=0.85,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True,
    )
    return response.strip()


def save_session(history, archetype_key, session_id):
    """Save a completed chat session to JSONL.

    Roles are INVERTED for training output:
    - Gradio "assistant" (subscriber) → saved as "user"
    - Gradio "user" (Jasmin/Shane)   → saved as "assistant"
    """
    if not history:
        return 'No messages to save.'

    flipped = {'user': 'assistant', 'assistant': 'user'}
    messages = []
    for msg in history:
        messages.append({
            'role': flipped[msg['role']],
            'content': msg['content'],
            'timestamp': datetime.now(timezone.utc).isoformat(),
        })

    session = {
        'messages': messages,
        'archetype': archetype_key,
        'turns': len([m for m in history if m['role'] == 'user']),
        'session_id': session_id,
    }

    with open(SESSIONS_FILE, 'a') as f:
        f.write(json.dumps(session) + '\n')

    turn_count = session['turns']
    return f'Session saved ({turn_count} turns, archetype={archetype_key})'


print('Subscriber bot logic ready.')

In [ ]:
# ── Cell 5: Gradio Chat UI ─────────────────────────────────────────
# Archetype dropdown + chat interface + save button.
# Shane types as Jasmin, bot responds as the selected subscriber type.

import gradio as gr

current_archetype = 'horny'
current_session_id = str(uuid.uuid4())[:8]


def on_archetype_change(archetype_key):
    """Reset session when archetype changes."""
    global current_archetype, current_session_id
    current_archetype = archetype_key
    current_session_id = str(uuid.uuid4())[:8]
    opener = ARCHETYPES[archetype_key]['opener']
    return [{'role': 'assistant', 'content': opener}]


def user_sends_message(user_message, history):
    """Handle Shane's message as Jasmin and generate subscriber reply."""
    if not history:
        history = []

    history.append({'role': 'user', 'content': user_message})

    subscriber_reply = generate_subscriber_message(
        assistant_message=user_message,
        history=history,
        archetype_key=current_archetype,
    )

    history.append({'role': 'assistant', 'content': subscriber_reply})
    return '', history


def on_save_click(history):
    """Save the current session to JSONL."""
    result = save_session(history, current_archetype, current_session_id)
    return result


def on_new_session(archetype_key):
    """Start a fresh session with the current archetype."""
    global current_session_id
    current_session_id = str(uuid.uuid4())[:8]
    opener = ARCHETYPES[archetype_key]['opener']
    return [{'role': 'assistant', 'content': opener}], ''


# Build the UI
archetype_choices = [(v['label'], k) for k, v in ARCHETYPES.items()]

with gr.Blocks(title='Subscriber Simulator') as demo:
    gr.Markdown('# Subscriber Simulator\nYou are **Jasmin**. The bot is a subscriber. Select an archetype and chat.')

    with gr.Row():
        archetype_dropdown = gr.Dropdown(
            choices=archetype_choices,
            value='horny',
            label='Subscriber Archetype',
            scale=2,
        )
        new_session_btn = gr.Button('New Session', scale=1)
        save_btn = gr.Button('Save Session', variant='primary', scale=1)

    chatbot = gr.Chatbot(
        value=[{'role': 'assistant', 'content': ARCHETYPES['horny']['opener']}],
        label='Chat',
        height=500,
    )
    msg_input = gr.Textbox(
        placeholder='Type as Jasmin...',
        label='Your message (as Jasmin)',
        show_label=False,
    )
    status_output = gr.Textbox(label='Status', interactive=False)

    # Event bindings
    archetype_dropdown.change(
        fn=on_archetype_change,
        inputs=[archetype_dropdown],
        outputs=[chatbot],
    )

    msg_input.submit(
        fn=user_sends_message,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot],
    )

    save_btn.click(
        fn=on_save_click,
        inputs=[chatbot],
        outputs=[status_output],
    )

    new_session_btn.click(
        fn=on_new_session,
        inputs=[archetype_dropdown],
        outputs=[chatbot, status_output],
    )

try:
    demo.launch(share=IS_COLAB)
except Exception:
    print('Share tunnel failed — falling back to local-only.')
    demo.launch(share=False)